In [ ]:
!pip install clipspy ipywidgets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.2/901.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.7 MB/s eta 0:00:00


In [ ]:
import clips

def construir_motor():
    env = clips.Environment()

    env.build("""
        (deftemplate alumno
            (slot nombre (type STRING))
            (slot promedio (type FLOAT))
            (slot porcentaje-faltas (type FLOAT))
            (slot tarea-completa (type SYMBOL) (allowed-symbols si no)))
    """)

    env.build("""
        (deftemplate resultado
            (slot nombre (type STRING))
            (slot estado (type STRING))
            (slot detalle (type STRING))
            (slot prioridad (type INTEGER)))
    """)

    # Regla 1: Desempeño Superior
    env.build("""
        (defrule regla-desempeno-superior
            (declare (salience 10))
            (alumno (nombre ?n) (promedio ?p&:(>= ?p 4.6)))
            =>
            (assert (resultado (nombre ?n) (estado "Desempeño Superior")
                                (detalle (str-cat "Promedio: " ?p " (Excelente)"))
                                (prioridad 10))))
    """)

    # Regla 2: Desempeño Alto
    env.build("""
        (defrule regla-desempeno-alto
            (declare (salience 10))
            (alumno (nombre ?n) (promedio ?p&:(and (>= ?p 4.0) (< ?p 4.6))))
            =>
            (assert (resultado (nombre ?n) (estado "Desempeño Alto")
                                (detalle (str-cat "Promedio: " ?p))
                                (prioridad 10))))
    """)

    # Regla 3: Desempeño Básico
    env.build("""
        (defrule regla-desempeno-basico
            (declare (salience 10))
            (alumno (nombre ?n) (promedio ?p&:(and (>= ?p 3.0) (< ?p 4.0))))
            =>
            (assert (resultado (nombre ?n) (estado "Desempeño Básico")
                                (detalle (str-cat "Promedio: " ?p " (Aprobado)"))
                                (prioridad 10))))
    """)

    # Regla 4: Desempeño Bajo / Reprobado
    env.build("""
        (defrule regla-desempeno-bajo
            (declare (salience 10))
            (alumno (nombre ?n) (promedio ?p&:(< ?p 3.0)))
            =>
            (assert (resultado (nombre ?n) (estado "Desempeño Bajo")
                                (detalle (str-cat "Promedio: " ?p " (Reprobado)"))
                                (prioridad 10))))
    """)

    # Regla 5: Reprobado por inasistencia (mayor prioridad, anula la nota)
    env.build("""
        (defrule regla-inasistencia
            (declare (salience 20))
            (alumno (nombre ?n) (porcentaje-faltas ?f&:(> ?f 20.0)))
            =>
            (assert (resultado (nombre ?n) (estado "Reprobado por Inasistencia")
                                (detalle (str-cat "Faltas: " ?f "% (supera el 20% permitido)"))
                                (prioridad 20))))
    """)

    # Regla 6: Pendiente por actividades de refuerzo
    env.build("""
        (defrule regla-recuperacion
            (declare (salience 15))
            (alumno (nombre ?n) (promedio ?p&:(>= ?p 3.0))
                                (tarea-completa no)
                                (porcentaje-faltas ?f&:(<= ?f 20.0)))
            =>
            (assert (resultado (nombre ?n) (estado "Pendiente - Actividades de Refuerzo")
                                (detalle (str-cat "Promedio: " ?p " pero con tareas incompletas"))
                                (prioridad 15))))
    """)

    return env


def evaluar(env, nombre, promedio, porcentaje_faltas, tarea_completa):
    """Ejecuta el motor y devuelve el resultado de MAYOR prioridad."""
    env.reset()
    env.assert_string(
        f'(alumno (nombre "{nombre}") (promedio {float(promedio)}) '
        f'(porcentaje-faltas {float(porcentaje_faltas)}) '
        f'(tarea-completa {"si" if tarea_completa else "no"}))'
    )
    env.run()

    resultados = [f for f in env.facts() if f.template.name == "resultado"]
    if not resultados:
        return None
    mejor = max(resultados, key=lambda r: r["prioridad"])
    return {"estado": mejor["estado"], "detalle": mejor["detalle"]}

# Construimos el motor una sola vez
env = construir_motor()
print("Motor de reglas construido correctamente. Reglas cargadas:", len(list(env.rules())))

Motor de reglas construido correctamente. Reglas cargadas: 6


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- Estilos CSS personalizados (tarjeta blanca) ---
display(HTML("""
<style>
.caja-blanca {
    background-color: #ffffff !important;
    border-radius: 14px !important;
    padding: 24px !important;
    box-shadow: 0 4px 14px rgba(0,0,0,0.12) !important;
    border: 1px solid #e0e0e0 !important;
    max-width: 480px;
}
.caja-blanca .widget-label { color: #1f3c58 !important; font-weight: 600; }

/* Campo de texto (Nombre): fondo claro */
.caja-blanca .widget-text input {
    background-color: #f1f4f8 !important;
    color: #1f2937 !important;
    border: 1px solid #cbd5e1 !important;
}
.caja-blanca .widget-text input:focus {
    background-color: #ffffff !important;
    border: 1px solid #1f3c58 !important;
}

/* Toggle buttons (Tareas completas): claro por defecto, oscuro al seleccionar */
.caja-blanca .widget-toggle-buttons button,
.caja-blanca .widget-toggle-buttons .jupyter-button {
    background-color: #eef1f5 !important;
    color: #1f2937 !important;
    border: 1px solid #cbd5e1 !important;
    font-weight: 500;
}
.caja-blanca .widget-toggle-buttons button:hover {
    background-color: #dde3ea !important;
}
.caja-blanca .widget-toggle-buttons button.mod-active,
.caja-blanca .widget-toggle-buttons button[aria-pressed="true"] {
    background-color: #1f3c58 !important;
    color: #ffffff !important;
    border: 1px solid #1f3c58 !important;
}

/* Números de los sliders (Promedio y Faltas): visibles, mismo color que el resto */
.caja-blanca .widget-readout {
    color: #1f3c58 !important;
    background: transparent !important;
    font-weight: 600;
}
</style>
"""))

COLORES = {
    "Desempeño Superior": "#1b5e20",
    "Desempeño Alto": "#2e7d32",
    "Desempeño Básico": "#f9a825",
    "Desempeño Bajo": "#c62828",
    "Reprobado por Inasistencia": "#8e0000",
    "Pendiente - Actividades de Refuerzo": "#ef6c00",
}

# --- Widgets de entrada ---
txt_nombre = widgets.Text(
    value="Juan Pérez",
    description="Nombre:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="380px"),
)

sld_promedio = widgets.FloatSlider(
    value=0, min=0.0, max=5.0, step=0.1,
    description="Promedio:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    readout_format=".1f",
)

sld_faltas = widgets.IntSlider(
    value=0, min=0, max=100, step=1,
    description="Faltas (%):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
)

chk_tarea = widgets.ToggleButtons(
    options=[("Sí", True), ("No", False)],
    value=True,
    description="Tareas completas:",
    style={"description_width": "120px"},
)

btn_evaluar = widgets.Button(
    description="Evaluar alumno",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="220px", height="42px", margin="10px 0 0 0"),
)

salida = widgets.Output()
salida_historial = widgets.Output()

historial = []

def on_click(b):
    nombre = txt_nombre.value.strip() or "Alumno"
    promedio = sld_promedio.value
    faltas = sld_faltas.value
    tarea_completa = chk_tarea.value

    resultado = evaluar(env, nombre, promedio, faltas, tarea_completa)

    with salida:
        clear_output()
        if resultado is None:
            display(HTML("<b>Sin resultado determinado.</b>"))
        else:
            color = COLORES.get(resultado["estado"], "#333")
            display(HTML(f"""
                <div style="border:1px solid #eee; border-radius:10px; padding:16px;
                            background:#fafafa; margin-top:14px;">
                    <div style="font-size:18px; font-weight:bold; color:{color};">
                        {resultado['estado']}
                    </div>
                    <div style="font-size:13px; color:#444; margin-top:6px;">
                        {resultado['detalle']}
                    </div>
                </div>
            """))

    historial.append({
        "Nombre": nombre, "Promedio": promedio, "Faltas %": faltas,
        "Tareas": "Sí" if tarea_completa else "No",
        "Estado": resultado["estado"] if resultado else "-",
    })

    with salida_historial:
        clear_output()
        import pandas as pd
        display(pd.DataFrame(historial[::-1]))

btn_evaluar.on_click(on_click)

titulo = widgets.HTML("<h3 style='color:#1f3c58; margin-top:0;'>📋 Evaluación del alumno</h3>")

fila_boton = widgets.HBox(
    [btn_evaluar],
    layout=widgets.Layout(justify_content="center", width="100%")
)

formulario = widgets.VBox(
    [titulo, txt_nombre, sld_promedio, sld_faltas, chk_tarea, fila_boton, salida],
    layout=widgets.Layout(padding="0")
)
formulario.add_class("caja-blanca")

display(formulario)

In [ ]:
display(HTML("""
<style>
.caja-historial {
    background-color: #ffffff !important;
    border-radius: 14px !important;
    padding: 20px !important;
    box-shadow: 0 4px 14px rgba(0,0,0,0.12) !important;
    border: 1px solid #e0e0e0 !important;
}

/* Texto de la tabla del historial: mismo tono que el resto de la interfaz */
.caja-historial table {
    color: #1f3c58 !important;
    background-color: #ffffff !important;
}
.caja-historial table th {
    color: #1f3c58 !important;
    background-color: #f1f4f8 !important;
    font-weight: 600 !important;
}
.caja-historial table td {
    color: #1f2937 !important;
    background-color: #ffffff !important;
}
</style>
"""))

titulo_hist = widgets.HTML("<h3 style='color:#1f3c58; margin-top:0;'>🗂️ Historial</h3>")
caja_hist = widgets.VBox([titulo_hist, salida_historial])
caja_hist.add_class("caja-historial")
display(caja_hist)